# 06 · Regularization — AtliQ churn

L1 demo, manual batch-norm, then base / dropout / L2 / BatchNorm1d on the churn CSV.

Source: `Model_Optimization_Regularization_Exercise`

The course dropout/BN templates have mismatched Linear sizes. Here every model is **6 → 32 → 16 → 1**, matching the written spec.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## Warm-up 1 — L1 term and its gradient

In [ ]:
weights = torch.tensor([0.5, -0.3, 0.8, -1.5], requires_grad=True)
l1_regularization = torch.sum(torch.abs(weights))
l1_regularization.backward()
print(f'L1 Regularization: {l1_regularization.item():.4f}')
print(f'Gradients: {weights.grad}')  # sign(w) → [1, -1, 1, -1]

## Warm-up 2 — manual batch norm (feature-wise)

In [ ]:
inputs = torch.tensor([[10.0, 20.0], [15.0, 25.0], [12.0, 18.0]])
mean = inputs.mean(dim=0)
std = inputs.std(dim=0, unbiased=False)
normalized = (inputs - mean) / (std + 1e-8)
print('Inputs:\n', inputs)
print('Mean:', mean)
print('Std:', std)
print('Normalized:\n', normalized)

## Churn dataset

In [ ]:
CSV_NAME = 'AtliQ_Churn_Prediction_Codebasics_DL.csv'
RAW = 'https://raw.githubusercontent.com/yash9614/lear_deepintodl/main/Model_Optimization_Regularization_Exercise/AtliQ_Churn_Prediction_Codebasics_DL.csv'
LOCALS = [CSV_NAME, f'../Model_Optimization_Regularization_Exercise/{CSV_NAME}', f'/content/{CSV_NAME}']

def load_csv(raw, locals_, name):
    try:
        return pd.read_csv(raw)
    except Exception:
        pass
    for p in locals_:
        if os.path.exists(p):
            return pd.read_csv(p)
    from google.colab import files
    print('Upload', name)
    up = files.upload()
    return pd.read_csv(next(iter(up)))

data = load_csv(RAW, LOCALS, CSV_NAME)
print(data.info())
X = data.drop('Churned', axis=1).values
y = data['Churned'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
print(X_train_tensor.shape, y_train_tensor.mean().item())

In [ ]:
class BaseModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(6, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.network(x)


class DropoutModel(nn.Module):
    def __init__(self, p=0.4):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(6, 32),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.network(x)


class BatchNormModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(6, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.layers(x)


def train_loop(model, optimizer, title, epochs=30, batch_size=32):
    loss_function = nn.BCELoss()
    model = model.to(device)
    Xtr, ytr = X_train_tensor.to(device), y_train_tensor.to(device)
    Xte, yte = X_test_tensor.to(device), y_test_tensor.to(device)
    n = len(Xtr)
    num_batches = n // batch_size
    train_hist, val_hist = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        perm = torch.randperm(n, device=device)
        for i in range(num_batches):
            idx = perm[i * batch_size:(i + 1) * batch_size]
            optimizer.zero_grad()
            pred = model(Xtr[idx])
            loss = loss_function(pred, ytr[idx])
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        train_hist.append(epoch_loss / num_batches)

        model.eval()
        with torch.no_grad():
            val_pred = model(Xte)
            val_loss = loss_function(val_pred, yte)
            val_hist.append(val_loss.item())
            acc = ((val_pred >= 0.5).float() == yte).float().mean().item()
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'{title}  ep {epoch+1:02d}  train {train_hist[-1]:.4f}  val {val_hist[-1]:.4f}  acc {acc:.3f}')

    plt.figure(figsize=(7, 3.5))
    plt.plot(range(1, epochs + 1), train_hist, label='Training Loss')
    plt.plot(range(1, epochs + 1), val_hist, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.show()
    return train_hist, val_hist

### Base model (no regularization)

In [ ]:
base = BaseModel()
opt = optim.SGD(base.parameters(), lr=0.01)
train_loop(base, opt, 'Base model')

### Dropout p=0.4

In [ ]:
drop = DropoutModel(p=0.4)
opt_d = optim.SGD(drop.parameters(), lr=0.01)
train_loop(drop, opt_d, 'Dropout p=0.4')

### L2 via `weight_decay=0.1` on a fresh base net

In [ ]:
l2_net = BaseModel()
opt_l2 = optim.SGD(l2_net.parameters(), lr=0.01, weight_decay=0.1)
train_loop(l2_net, opt_l2, 'L2 weight_decay=0.1')

### BatchNorm1d

In [ ]:
bn = BatchNormModel()
opt_bn = optim.SGD(bn.parameters(), lr=0.01)
train_loop(bn, opt_bn, 'BatchNorm1d')